In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

In [7]:
# For reproducibility
np.random.seed(42)

In [8]:
df = pd.read_csv("Diabetes_and_LifeStyle_Dataset_.csv")

In [9]:
# Data Cleaning
# Check for duplicates even if there is none
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")
if duplicate_count > 0:
    df = df.drop_duplicates()

Number of duplicate rows: 0


In [10]:
# Handle missing values even if there is none: for future record additions
# Numerical Columns: impute with median
# Categorical Columns: impute with mode
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = df.select_dtypes(include=['object']).columns.tolist()

In [11]:
# Remove target variables from feature lists
target = 'diabetes_stage'
if 'diagnosed_diabetes' in numerical_features:
    numerical_features.remove('diagnosed_diabetes')
if target in categorical_features:
    categorical_features.remove(target)

In [13]:
# Feature engineering

# Create age groups
df['age_group'] = pd.cut(df['Age'], bins=[0, 30, 45, 60, 100], labels=['Young', 'Middle-aged', 'Senior', 'Elderly'])

# BMI categories
df['bmi_category'] = pd.cut(df['bmi'], bins=[0, 18.5, 25, 30, 100], labels=['Underweight', 'Normal', 'Overweight', 'Obese'])

# Blood pressure categories
df['bp_category'] = 'Normal'
df.loc[(df['systolic_bp'] >= 140) | (df['diastolic_bp'] >= 90), 'bp_category'] = 'Hypertension'
df.loc[(df['systolic_bp'] >= 120) & (df['systolic_bp'] < 140) | (df['diastolic_bp'] >= 80) & (df['diastolic_bp'] < 90), 'bp_category'] = 'Prehypertension'

# Add these new features to categorical list
categorical_features.extend(['age_group', 'bmi_category', 'bp_category'])

In [14]:
# Prepare data for modeling
# Seperate features and target
X = df.drop([target, 'diagnosed_diabetes'], axis=1)  # Removes both target variables ('diagnosed_diabetes' represents a different but related prediction task from the main target 'diabetes_stage')
y_class = df[target] # Classification target
y_binary = df['diagnosed_diabetes'] # Binary target

# Define preprocessing for numerical features
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Define preprocessing for categorical features
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Encode the target variable for classification
label_encoder = LabelEncoder()
y_class_encoded = label_encoder.fit_transform(y_class)
print(f"Target classes: {list(label_encoder.classes_)}")

Target classes: ['Gestational', 'No Diabetes', 'Pre-Diabetes', 'Type 1', 'Type 2']


In [15]:
# Split data

# Split data into training and testing sets
X_train, X_test, y_class_train, y_class_test, y_binary_train, y_binary_test = train_test_split(
    X, y_class_encoded, y_binary, test_size=0.2, random_state=42, stratify=y_class_encoded
)

# Apply preprocessing
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Get feature names after transformation
ohe_feature_names = []
if categorical_features:
    ohe_feature_names = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_features)
feature_names = numerical_features + list(ohe_feature_names)

# Convert to Dataframe for better visualization
X_train_processed_df = pd.DataFrame(X_train_processed, columns=feature_names)

In [16]:
# Summary of prepared data
print(f"\nTraining set shape: {X_train_processed.shape}")
print(f"Testing set shape: {X_test_processed.shape}")
print(f"\nClass distribution in training set:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {cls}: {np.sum(y_class_train == i)} ({np.sum(y_class_train == i)/len(y_class_train):.2%})")

print(f"\nClass distribution in testing set:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {cls}: {np.sum(y_class_test == i)} ({np.sum(y_class_test == i)/len(y_class_test):.2%})")

print(f"\nBinary target distribution in training set:")
print(f"  0: {np.sum(y_binary_train == 0)} ({np.sum(y_binary_train == 0)/len(y_binary_train):.2%})")
print(f"  1: {np.sum(y_binary_train == 1)} ({np.sum(y_binary_train == 1)/len(y_binary_train):.2%})")

# Display first few rows of processed training data
print("\nFirst 5 rows of processed training data:")
print(X_train_processed_df.head())


Training set shape: (77837, 58)
Testing set shape: (19460, 58)

Class distribution in training set:
  Gestational: 214 (0.27%)
  No Diabetes: 6189 (7.95%)
  Pre-Diabetes: 24810 (31.87%)
  Type 1: 94 (0.12%)
  Type 2: 46530 (59.78%)

Class distribution in testing set:
  Gestational: 53 (0.27%)
  No Diabetes: 1548 (7.95%)
  Pre-Diabetes: 6203 (31.88%)
  Type 1: 23 (0.12%)
  Type 2: 11633 (59.78%)

Binary target distribution in training set:
  0: 31123 (39.98%)
  1: 46714 (60.02%)

First 5 rows of processed training data:
        Age  alcohol_consumption_per_week  physical_activity_minutes_per_week  \
0 -1.432676                     -0.709991                           -0.983056   
1 -0.205414                      0.697963                           -0.309363   
2  0.569700                     -0.709991                           -0.971237   
3  0.505107                     -0.006014                           -0.096618   
4 -0.592970                      0.697963                            